# Experiment 6: GloVe Word Embeddings and N-gram Language Modelling

**Tasks from the lab manual:**
1. Implement GloVe embeddings from scratch (without a pretrained model) and find words most similar to `king`.
2. Load pretrained GloVe embeddings using Gensim and find words most similar to `king`.
3. Implement a statistical N-gram language model from scratch to predict the next word.

The first two tasks use the supplied `glove_corpus.txt`. The third task uses the exact sample sentences printed in Experiment-6.docx.

**Colab setup:** Open this notebook in Google Colab (`File → Upload notebook`). Run the cells from top to bottom. The pretrained-model cell downloads a model and therefore needs an internet connection.

## 1. Load the supplied GloVe corpus

The supplied `glove_corpus.txt` contains 24 short sentences. For a lab demonstration this is fine, but embeddings trained from such a small corpus will be limited and may not produce strong semantic relationships.

In [ ]:
# Corpus supplied with the experiment
corpus_text = """
The king rules the kingdom.
The queen rules the kingdom.
The king is a powerful man.
The queen is a wise woman.
The man is strong and brave.
The woman is intelligent and kind.
The king and queen rule together.
The man and woman live in the kingdom.
The prince is the son of the king.
The princess is the daughter of the queen.
The king lives in a royal palace.
The queen lives in a royal palace.
A doctor treats a patient.
A teacher teaches students in a school.
Students read books and learn new things.
A computer processes data and information.
A computer scientist develops software.
The scientist conducts research in a laboratory.
The cat drinks milk.
The dog eats food.
The cat and dog live in a house.
The sun gives light and heat.
The moon shines at night.
The earth revolves around the sun.
"""

import re
sentences = [s.strip().lower() for s in corpus_text.strip().splitlines() if s.strip()]
tokenized_corpus = [re.findall(r"\\b[a-z]+\\b", s) for s in sentences]

print("Number of sentences:", len(tokenized_corpus))
print("First tokenized sentence:", tokenized_corpus[0])
print("Vocabulary size:", len(set(w for sent in tokenized_corpus for w in sent)))

## 2. GloVe from scratch (no pretrained model)

GloVe learns vectors from global word-word co-occurrence counts. Words appearing near each other contribute to a co-occurrence matrix. The model learns two sets of vectors and biases by minimizing:

\[
J = \sum_{i,j} f(X_{u})\left(w_i^T\tilde{w}_j+b_i+\tilde{b}_j-\log X_{ij}\right)^2
\]

Here, `X_ij` is the number of times context word `j` appears near target word `i`. This cell uses a small context window and trains the vectors with gradient descent.

In [ ]:
import numpy as np
from collections import Counter, defaultdict

# Build vocabulary
word_counts = Counter(w for sent in tokenized_corpus for w in sent)
vocab = sorted(word_counts)
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}
V = len(vocab)

# Build the symmetric co-occurrence matrix
window_size = 3
cooccur = defaultdict(float)

for sent in tokenized_corpus:
    ids = [word_to_id[w] for w in sent]
    for center_pos, center_id in enumerate(ids):
        start = max(0, center_pos - window_size)
        end = min(len(ids), center_pos + window_size + 1)
        for context_pos in range(start, end):
            if context_pos == center_pos:
                continue
            context_id = ids[context_pos]
            distance = abs(context_pos - center_pos)
            cooccur[(center_id, context_id)] += 1.0 / distance

pairs = [(i, j, x) for (i, j), x in cooccur.items()]
print("Vocabulary size:", V)
print("Non-zero co-occurrence pairs:", len(pairs))

In [ ]:
# Train GloVe vectors using weighted least-squares objective
np.random.seed(42)
embedding_dim = 50
W = (np.random.rand(V, embedding_dim) - 0.5) / embedding_dim
W_context = (np.random.rand(V, embedding_dim) - 0.5) / embedding_dim
b = np.zeros(V)
b_context = np.zeros(V)

x_max = 10.0
alpha = 0.75
learning_rate = 0.05
epochs = 300

for epoch in range(epochs):
    total_loss = 0.0
    for i, j, x in pairs:
        weight = (x / x_max) ** alpha if x < x_max else 1.0
        log_x = np.log(x)
        diff = np.dot(W[i], W_context[j]) + b[i] + b_context[j] - log_x
        loss = weight * diff * diff
        total_loss += 0.5 * loss

        grad = weight * diff
        grad_w = grad * W_context[j].copy()
        grad_wc = grad * W[i].copy()

        W[i] -= learning_rate * grad_w
        W_context[j] -= learning_rate * grad_wc
        b[i] -= learning_rate * grad
        b_context[j] -= learning_rate * grad

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs}, loss = {total_loss:.4f}")

# Standard GloVe representation combines target and context vectors
glove_vectors = W + W_context
print("Training complete. Embedding matrix shape:", glove_vectors.shape)

In [ ]:
# Find words most similar to 'king' using cosine similarity
def most_similar_custom(query, topn=5):
    if query not in word_to_id:
        return f"'{query}' is not present in the training vocabulary."
    q = glove_vectors[word_to_id[query]]
    q_norm = np.linalg.norm(q)
    scores = glove_vectors @ q / (np.linalg.norm(glove_vectors, axis=1) * q_norm + 1e-10)
    scores[word_to_id[query]] = -1
    best_ids = np.argsort(scores)[::-1][:topn]
    return [(id_to_word[i], float(scores[i])) for i in best_ids]

print("Words most similar to king:")
for word, score in most_similar_custom("king", topn=5):
    print(f"{word:15s} {score:.4f}")

## 3. Pretrained GloVe model using Gensim

This section uses Gensim's pretrained `glove-wiki-gigaword-100` vectors. These embeddings were trained on a much larger external corpus, rather than on the small supplied file. The query word `king` is looked up in the pretrained vocabulary.

In [ ]:
!pip -q install gensim

In [ ]:
import gensim.downloader as api

# Downloads the pretrained GloVe vectors (100 dimensions)
glove_pretrained = api.load("glove-wiki-gigaword-100")

print("Pretrained model loaded.")
print("Vector dimension:", glove_pretrained.vector_size)
print("Most similar words to 'king':")
for word, score in glove_pretrained.most_similar("king", topn=10):
    print(f"{word:20s} {score:.4f}")

## 4. N-gram statistical language model from scratch

Use the sample corpus exactly as shown in the lab manual. The model below supports unigram, bigram, and trigram prediction. It uses add-one (Laplace) smoothing so unseen next-word candidates receive a non-zero probability.

For an N-gram model, the next-word probability is estimated from the previous `n-1` words.

In [ ]:
lm_text = """
natural language processing is a branch of artificial intelligence.
natural language processing enables computers to understand human language.
machine learning is a branch of artificial intelligence.
deep learning is a part of machine learning.
language models learn patterns from text data.
a language model predicts the next word in a sentence.
natural language processing uses language models.
artificial intelligence includes machine learning and deep learning.
"""

lm_sentences = [s.strip().lower() for s in lm_text.strip().splitlines() if s.strip()]
lm_tokens = [re.findall(r"[a-z]+", s) for s in lm_sentences]

print("Number of LM sentences:", len(lm_tokens))
for sent in lm_tokens:
    print(sent)

In [ ]:
from collections import Counter

class NGramLanguageModel:
    def __init__(self, n=3):
        if n < 1:
            raise ValueError("n must be at least 1")
        self.n = n
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.vocabulary = set()

    def fit(self, tokenized_sentences):
        self.vocabulary = {w for sent in tokenized_sentences for w in sent}
        self.vocabulary.add("<UNK>")

        for sent in tokenized_sentences:
            tokens = ["<s>"] * (self.n - 1) + sent + ["</s>"]
            for i in range(self.n - 1, len(tokens)):
                ngram = tuple(tokens[i - self.n + 1:i + 1]) if self.n > 1 else (tokens[i],)
                context = ngram[:-1]
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1

    def predict_next(self, context_words, top_k=5):
        context_len = self.n - 1
        if context_len == 0:
            context = ()
        else:
            cleaned = [w.lower() for w in context_words.split()]
            cleaned = cleaned[-context_len:]
            context = tuple(cleaned)
            if len(context) < context_len:
                context = tuple(["<s>"] * (context_len - len(context)) + list(context))

        V = len(self.vocabulary) + 1  # include end-of-sentence token
        candidates = list(self.vocabulary) + ["</s>"]
        denominator = self.context_counts[context] + V
        scored = []
        for word in candidates:
            ngram = context + (word,)
            probability = (self.ngram_counts[ngram] + 1) / denominator
            scored.append((word, probability))
        return sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]

# Train trigram model
trigram_model = NGramLanguageModel(n=3)
trigram_model.fit(lm_tokens)

context = "natural language"
print(f"Context: {context!r}")
print("Predicted next words:")
for word, probability in trigram_model.predict_next(context, top_k=5):
    print(f"{word:15s} probability = {probability:.4f}")

In [ ]:
# Compare unigram, bigram and trigram models on a context
test_context = "machine learning"

for n in (1, 2, 3):
    model = NGramLanguageModel(n=n)
    model.fit(lm_tokens)
    print(f"\\nTop predictions for {n}-gram model | context: {test_context!r}")
    for word, probability in model.predict_next(test_context, top_k=5):
        print(f"{word:15s} {probability:.4f}")

## 5. Conclusion / observations to write in the record

1. The from-scratch GloVe implementation builds a word co-occurrence matrix and learns word vectors from the supplied corpus.
2. The nearest words returned for `king` depend on the corpus size, context-window size, embedding dimension, and training iterations. Since the supplied corpus is very small, results may be unstable.
3. The pretrained Gensim GloVe model uses vectors learned from a large external text collection, so its similarity results are generally more meaningful.
4. The N-gram model estimates the next word using frequencies of preceding words. Unigram uses no context, bigram uses one preceding word, and trigram uses two.
5. Laplace smoothing prevents zero probabilities for words not observed after a particular context.

**Important:** Record the actual output produced by your Colab run. Do not copy expected similarity rankings as fixed results, because training and corpus details can change them.